# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MbaliThusi/Mbali-Flyrank-intern-rep./blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
print(df.shape)

Saving content_refresh_anonymized.csv to content_refresh_anonymized (2).csv
(30000, 44)


In [14]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
print(df.shape)


Saving content_refresh_anonymized.csv to content_refresh_anonymized (3).csv
(30000, 44)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:
import numpy as np

# Label: did clicks decline in the last 30 days vs the previous 30?
base = df[df["clicks_prev_30d"] > 0].copy()   # decline is undefined with 0 prior clicks
y = (base["clicks_last_30d"] < base["clicks_prev_30d"]).astype(int)

cat_cols = ["content_type", "main_intent", "competition_level", "provider_used", "model_used"]
num_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
            "content_age_days", "days_since_last_update",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]

X = base[num_cols + cat_cols].copy()

# Engineered features
X["chars_per_word"] = base["char_count"] / base["word_count"].replace(0, np.nan)
X["prev_ctr"] = base["clicks_prev_30d"] / base["impressions_prev_30d"].replace(0, np.nan)
X["log_clicks_prev"] = np.log1p(base["clicks_prev_30d"])
X["log_impr_prev"] = np.log1p(base["impressions_prev_30d"])
X = X.replace([np.inf, -np.inf], np.nan)

# Missing values: flag, then fill numerics with the median
for c in list(X.select_dtypes("number").columns):
    if X[c].isna().any():
        X[c + "_was_missing"] = X[c].isna().astype(int)
        X[c] = X[c].fillna(X[c].median())

# Categoricals: explicit "unknown", then one-hot
X[cat_cols] = X[cat_cols].fillna("unknown")
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape, "| positive rate:", y.mean().round(3))

(11759, 35) | positive rate: 0.579


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [17]:
from sklearn.metrics import roc_auc_score

# Test every feature on its own: a legit feature rarely scores very high alone
rows = []
for c in X.columns:
    try:
        auc = roc_auc_score(y, X[c].astype(float))
        rows.append((c, max(auc, 1 - auc)))
    except Exception:
        pass
leak = pd.DataFrame(rows, columns=["feature", "single_feature_auc"]) \
         .sort_values("single_feature_auc", ascending=False)
print(leak.head(10))     # anything above ~0.8 deserves suspicion

# Test the columns I plan to exclude: these should score very high, proving the leak
suspects = ["clicks_last_30d", "clicks_90d", "trend_pct", "ctr"]
for c in suspects:
    s = base[c].replace([np.inf, -np.inf], np.nan).fillna(0)
    auc = roc_auc_score(y, s)
    print(c, round(max(auc, 1 - auc), 3))

                              feature  single_feature_auc
11                           prev_ctr            0.673070
8                     clicks_prev_30d            0.560962
12                    log_clicks_prev            0.560962
31  model_used_gemini-3-flash-preview            0.555384
13                      log_impr_prev            0.555092
7                impressions_prev_30d            0.555092
10                     chars_per_word            0.550071
3                          word_count            0.536458
4                          char_count            0.527356
26              competition_level_LOW            0.521137
clicks_last_30d 0.699
clicks_90d 0.574
trend_pct 0.652
ctr 0.522


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [18]:
excluded = {
    "clicks_last_30d, impressions_last_30d, sessions_last_30d": "the outcome window itself",
    "*_90d metrics (clicks, impressions, sessions, pageviews, users, engaged, ai, scroll)": "90d includes the last 30d, so it contains the future",
    "trend_direction, trend_pct": "computed directly from the label",
    "ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct": "computed over the 90d window, so partly future",
    "impression_tier, position_tier": "bucketed versions of 90d metrics",
    "days_with_impressions, days_with_sessions": "counted over the 90d window",
    "age_tier, age_tier_order, freshness_tier, word_count_tier, char_count_tier": "redundant bucketings of numeric columns already used",
    "client_id, content_id": "identifiers; keep for grouped splits, not as inputs",
}
for k, v in excluded.items():
    print(f"- {k}: {v}")

- clicks_last_30d, impressions_last_30d, sessions_last_30d: the outcome window itself
- *_90d metrics (clicks, impressions, sessions, pageviews, users, engaged, ai, scroll): 90d includes the last 30d, so it contains the future
- trend_direction, trend_pct: computed directly from the label
- ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct: computed over the 90d window, so partly future
- impression_tier, position_tier: bucketed versions of 90d metrics
- days_with_impressions, days_with_sessions: counted over the 90d window
- age_tier, age_tier_order, freshness_tier, word_count_tier, char_count_tier: redundant bucketings of numeric columns already used
- client_id, content_id: identifiers; keep for grouped splits, not as inputs


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.